# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-9862873/FlyRank-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the validated model output (ML-08, ML-09) into a content action playbook: a ranked queue with reason codes, the intended use and limits, human-review rules, monitoring triggers, and the exports my paper will reuse next week.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `writing-honest-claims` + `flyrank/flyrank-data` for this task.

## 0. Setup — same data, same split as ML-08 and ML-09

I retrain the Random Forest with the same features, the same client-grouped split, and the same seed as the modeling notebook, so every number in this playbook traces back to the same receipts. Nothing new is tuned here.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import json
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score, f1_score
import warnings
warnings.filterwarnings('ignore')

REPO_ROOT = Path('../..')
OUTPUTS_DIR = REPO_ROOT / 'work' / 'outputs'
FIGURES_DIR = REPO_ROOT / 'work' / 'figures'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(REPO_ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'content_age_days',
    'days_since_last_update', 'word_count'
]
df['has_word_count'] = df['word_count'].notna().astype(int)
df['has_position'] = (df['avg_position'] > 0).astype(int)
feature_cols += ['has_word_count', 'has_position']

X = df[feature_cols].fillna(0)
y = df['is_declining_label']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Grouped split (the honest numbers)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]

# Random split, for the memorization gap (same as ML-09)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_r = StandardScaler()
X_train_r_s = scaler_r.fit_transform(X_train_r)
X_test_r_s = scaler_r.transform(X_test_r)

lr_r = LogisticRegression(random_state=42, max_iter=1000)
lr_r.fit(X_train_r_s, y_train_r)
lr_r_prob = lr_r.predict_proba(X_test_r_s)[:, 1]

rf_r = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_r.fit(X_train_r, y_train_r)
rf_r_prob = rf_r.predict_proba(X_test_r)[:, 1]

# Baseline rule on the same test rows (kept as the comparison column)
test_df = df.iloc[te_idx].copy()
baseline_score_test = (
    (test_df['days_since_last_update'] >= 180).astype(int)
    * (test_df['impressions_90d'] >= 500).astype(int)
    * test_df['impressions_90d']
)

metrics = {
    'base_rate_overall': float(y.mean()),
    'base_rate_test': float(y_test.mean()),
    'grouped': {
        'n_test': int(len(y_test)),
        'n_test_clients': int(groups.iloc[te_idx].nunique()),
        'lr_auc': float(roc_auc_score(y_test, lr_prob)),
        'rf_auc': float(roc_auc_score(y_test, rf_prob)),
        'lr_f1': float(f1_score(y_test, lr.predict(X_test))),
        'rf_f1': float(f1_score(y_test, rf.predict(X_test))),
        'baseline_p10': float(precision_at_k(baseline_score_test.values, y_test.values, 10)),
        'lr_p10': float(precision_at_k(lr_prob, y_test.values, 10)),
        'rf_p10': float(precision_at_k(rf_prob, y_test.values, 10)),
        'baseline_p20': float(precision_at_k(baseline_score_test.values, y_test.values, 20)),
        'lr_p20': float(precision_at_k(lr_prob, y_test.values, 20)),
        'rf_p20': float(precision_at_k(rf_prob, y_test.values, 20)),
        'baseline_p50': float(precision_at_k(baseline_score_test.values, y_test.values, 50)),
        'lr_p50': float(precision_at_k(lr_prob, y_test.values, 50)),
        'rf_p50': float(precision_at_k(rf_prob, y_test.values, 50)),
    },
    'random': {
        'lr_auc': float(roc_auc_score(y_test_r, lr_r_prob)),
        'rf_auc': float(roc_auc_score(y_test_r, rf_r_prob)),
    },
}
metrics['memorization_gap_auc'] = {
    'lr': round(metrics['random']['lr_auc'] - metrics['grouped']['lr_auc'], 4),
    'rf': round(metrics['random']['rf_auc'] - metrics['grouped']['rf_auc'], 4),
}

print(f'Rows: {len(df):,}, base rate (declining): {metrics["base_rate_overall"]:.1%}')
print(f'Grouped test: {metrics["grouped"]["n_test"]:,} rows, {metrics["grouped"]["n_test_clients"]} unseen clients')
print(f'RF grouped: AUC {metrics["grouped"]["rf_auc"]:.3f}, P@10 {metrics["grouped"]["rf_p10"]:.2f}, P@20 {metrics["grouped"]["rf_p20"]:.2f}')
print(f'Memorization gap (random AUC - grouped AUC): RF {metrics["memorization_gap_auc"]["rf"]:.3f}, LR {metrics["memorization_gap_auc"]["lr"]:.3f}')

Rows: 30,000, base rate (declining): 54.2%
Grouped test: 6,163 rows, 7 unseen clients
RF grouped: AUC 0.603, P@10 0.70, P@20 0.70
Memorization gap (random AUC - grouped AUC): RF 0.154, LR 0.082


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The model ranks every page by its probability of declining. The **action** on each row does not come from the model alone — it comes from a short reason code a human can check against the page in two minutes. The model decides *priority*; the rule decides *what kind of look it needs*.

| Reason code | Rule (all observed signals) | Action |
|---|---|---|
| `insufficient_signal` | impressions_90d < 100 | NO_ACTION — too little traffic to separate decline from noise |
| `stale_visible` | stale >= 180d AND impressions >= 500 | REFRESH — the baseline rule from ML-07, kept as my honesty anchor |
| `high_traffic_weak_position` | impressions >= 1000 AND position >= 20 (page 2+) | REOPTIMIZE — visible but ranking weak; title/description first |
| `everything else` | the rest | REVIEW — model unsure, a human decides |

The baseline rule stays a comparison column in the export, exactly as ML-09 taught me: a queue that hides the rule it was checked against is a queue you cannot trust.

In [2]:
RF_FULL = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
RF_FULL.fit(X, y)
full_prob = RF_FULL.predict_proba(X)[:, 1]

baseline_rule_flag = ((df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)).astype(int)
baseline_full_score = baseline_rule_flag * df['impressions_90d']

def assign_reason_code(row):
    if row['impressions_90d'] < 100:
        return 'insufficient_signal', 'NO_ACTION'
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        return 'stale_visible', 'REFRESH'
    if row['impressions_90d'] >= 1000 and row['avg_position'] >= 20:
        return 'high_traffic_weak_position', 'REOPTIMIZE'
    return 'moderate_traffic_overlap', 'REVIEW'

codes = df.apply(assign_reason_code, axis=1, result_type='expand')
df['reason_code'] = codes[0]
df['action'] = codes[1]
df['model_score'] = full_prob
df['baseline_score'] = baseline_full_score

queue = df.sort_values('model_score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

print('Queue actions:')
print(queue['action'].value_counts().to_string())
print()
print('Reason codes:')
print(queue['reason_code'].value_counts().to_string())
print()
print('Top 10 of the queue:')
print(queue[['rank', 'content_id', 'action', 'reason_code', 'model_score',
             'impressions_90d', 'days_since_last_update', 'avg_position']].head(10).to_string(index=False))

Queue actions:
action
REVIEW        18471
NO_ACTION      7994
REOPTIMIZE     3518
REFRESH          17

Reason codes:
reason_code
moderate_traffic_overlap      18471
insufficient_signal            7994
high_traffic_weak_position     3518
stale_visible                    17

Top 10 of the queue:
 rank           content_id     action                reason_code  model_score  impressions_90d  days_since_last_update  avg_position
    1 content_8e32f425a490     REVIEW   moderate_traffic_overlap     0.862881             1767                      20           8.7
    2 content_9ef8e61550e9     REVIEW   moderate_traffic_overlap     0.857956             2812                      20           8.8
    3 content_48914a9c6eb2     REVIEW   moderate_traffic_overlap     0.856874             1559                      20           6.0
    4 content_09100d409cfc REOPTIMIZE high_traffic_weak_position     0.856439            48746                      20          40.0
    5 content_9d94766abf35     REVIEW   

### Archetype to action mapping

Three content archetypes live in this portfolio. They do not decline at the same rate, and one of them has no keyword data at all (per the data dictionary), so a refresh decision there rests on staleness and volume alone.

In [3]:
archetype = queue.groupby('content_type').agg(
    pages=('content_id', 'count'),
    decline_rate=('is_declining_label', 'mean'),
    median_impressions=('impressions_90d', 'median'),
    median_staleness=('days_since_last_update', 'median'),
).round(3)
print('Archetype summary (whole portfolio):')
print(archetype.to_string())
print()
print('Action mix within each archetype:')
print(pd.crosstab(queue['content_type'], queue['action']).to_string())
print()
print('Reading: keyword articles carry the volume (27,207 pages). Feedly articles have no')
print('keyword data at all, so the model judges them on traffic and staleness only.')
print('Comparison articles number 697 — too few to automate anything at the archetype level.')

Archetype summary (whole portfolio):
                    pages  decline_rate  median_impressions  median_staleness
content_type                                                                 
comparison article    697         0.572               107.0              20.0
feedly article       2096         0.287                 4.0              20.0
keyword article     27207         0.561               955.0              22.0

Action mix within each archetype:
action              NO_ACTION  REFRESH  REOPTIMIZE  REVIEW
content_type                                              
comparison article        331        0           4     362
feedly article           1744        0           9     343
keyword article          5919       17        3505   17766

Reading: keyword articles carry the volume (27,207 pages). Feedly articles have no
keyword data at all, so the model judges them on traffic and staleness only.
Comparison articles number 697 — too few to automate anything at the archetype lev

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** A content lead opens the queue once a week and works top-down. Each row says "this page looks worth reviewing first, because it is flagged `stale_visible` and still earns 60k impressions" — not "refreshing this page will restore its rankings." I have no intervention data, so the playbook is decision-support, never a guarantee.

**Limits, with the numbers attached:**

1. **The model is weak on unseen clients.** Grouped AUC is 0.603 (barely above the 0.5 coin flip). On a random split it looked like 0.757 — the 0.154 gap is memorization, and the grouped number is the honest one (ML-09).
2. **The features overlap the label window.** The 90-day aggregates include the last 30 days the label is measured in. Disclosed as LOW severity in ML-09, but it stays disclosed.
3. **Decay is not monotonic.** The paper's freshness story assumes older is worse. My own data disagrees: the 91-180d bucket declines most (61.1%), the 181-365d bucket declines less (46.7%). The baseline's 180d threshold sits right at the edge of a sparse region.

In [4]:
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[-1, 90, 180, 365, 100000],
    labels=['0-90d', '91-180d', '181-365d', '365d+'],
)
decay = df.groupby('staleness_bucket', observed=True).agg(
    pages=('content_id', 'count'),
    decline_rate=('is_declining_label', 'mean'),
).round(3)
print('Decline rate by staleness bucket:')
print(decay.to_string())
print()
print('The 181-365d bucket has only 169 pages and the 365d+ bucket has 5 —')
print('small n, so those two rates are noise, not signal. The honest reading:')
print('decline peaks around 3-6 months stale, and the data cannot say what happens after.')

Decline rate by staleness bucket:
                  pages  decline_rate
staleness_bucket                     
0-90d             20655         0.512
91-180d            9171         0.611
181-365d            169         0.467
365d+                 5         0.600

The 181-365d bucket has only 169 pages and the 365d+ bucket has 5 —
small n, so those two rates are noise, not signal. The honest reading:
decline peaks around 3-6 months stale, and the data cannot say what happens after.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before any REFRESH is touched, a human checks three things:**
1. Does the page still exist in the CMS, and is it still meant to rank?
2. Is there an obvious outside cause for the decline (seasonality, a campaign ended, the product it describes was retired)?
3. Does the page have brand or legal constraints on editing?

**Never automated — no exceptions in this playbook:**
- Publishing an edit. The queue recommends; a person writes and approves.
- Deleting or redirecting a page.
- Anything on rows with `avg_position = 0` (1,205 rows — the dictionary says that means *no data*, not rank zero).
- The 365d+ staleness bucket (n=5) and the 181-365d bucket (n=169) — too sparse to act on without individual eyes.
- Archetype-level rules for comparison articles (n=697) — one wrong rule moves a meaningful share of that archetype.

In [5]:
REFRESH_slice = queue[queue['action'] == 'REFRESH']

guardrail_checks = {
    'refresh_rows_all_stale_180plus': bool((REFRESH_slice['days_since_last_update'] >= 180).all()),
    'refresh_rows_all_visible_500plus': bool((REFRESH_slice['impressions_90d'] >= 500).all()),
    'no_position_zero_rows_in_refresh': bool((REFRESH_slice['avg_position'] > 0).all()),
    'no_365dplus_rows_in_refresh': bool((REFRESH_slice['days_since_last_update'] <= 365).all()),
}
print('Guardrail checks on the REFRESH slice:')
for name, passed in guardrail_checks.items():
    print(f'  {"PASS" if passed else "FAIL"}  {name}')
print()
print('Human review queue — top 20 rows a person should open this week:')
review_cols = ['rank', 'content_id', 'content_type', 'action', 'reason_code',
               'model_score', 'impressions_90d', 'days_since_last_update',
               'avg_position', 'ctr']
print(queue[review_cols].head(20).round(3).to_string(index=False))

Guardrail checks on the REFRESH slice:
  PASS  refresh_rows_all_stale_180plus
  PASS  refresh_rows_all_visible_500plus
  PASS  no_position_zero_rows_in_refresh
  PASS  no_365dplus_rows_in_refresh

Human review queue — top 20 rows a person should open this week:
 rank           content_id    content_type     action                reason_code  model_score  impressions_90d  days_since_last_update  avg_position  ctr
    1 content_8e32f425a490 keyword article     REVIEW   moderate_traffic_overlap        0.863             1767                      20           8.7 0.06
    2 content_9ef8e61550e9 keyword article     REVIEW   moderate_traffic_overlap        0.858             2812                      20           8.8 0.04
    3 content_48914a9c6eb2 keyword article     REVIEW   moderate_traffic_overlap        0.857             1559                      20           6.0 0.06
    4 content_09100d409cfc keyword article REOPTIMIZE high_traffic_weak_position        0.856            48746            

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Four signals, checked whenever a new labeled batch arrives:

1. **Base rate shift.** Today 54.2% of pages decline. If a fresh batch sits more than 5 points away, the world changed under the model — retrain before trusting the queue.
2. **Ranking quality drop.** If precision@20 on the newest labeled slice falls below 0.5, the queue's top is no longer worth a human's time.
3. **Feature drift.** If median impressions_90d moves more than 30% from today's value, the traffic distribution shifted — scores need refitting, not just recalibrating.
4. **New-client cohort.** Any client with under 90 days of history gets flagged out of the queue until it has a full window. A third of clients in this panel have little usable history — they would otherwise pollute the top of the list.

In [6]:
monitoring = {
    'checked_against': 'starter dataset v30k, 2026 snapshot',
    'base_rate_current': round(float(y.mean()), 4),
    'base_rate_retrain_trigger': 'shift > 0.05 absolute on a fresh labeled batch',
    'p20_floor': 0.5,
    'impressions_median_current': float(df['impressions_90d'].median()),
    'impressions_drift_trigger': 'median moves > 30% from current',
    'new_client_rule': 'clients with < 90 days history are excluded until a full window exists',
    'retrain_cadence': 'when any trigger fires, or every 6 months whichever comes first',
}
print('Monitoring thresholds (these go into the metrics export):')
for key, value in monitoring.items():
    print(f'  {key}: {value}')

Monitoring thresholds (these go into the metrics export):
  checked_against: starter dataset v30k, 2026 snapshot
  base_rate_current: 0.5421
  base_rate_retrain_trigger: shift > 0.05 absolute on a fresh labeled batch
  p20_floor: 0.5
  impressions_median_current: 731.0
  impressions_drift_trigger: median moves > 30% from current
  new_client_rule: clients with < 90 days history are excluded until a full window exists
  retrain_cadence: when any trigger fires, or every 6 months whichever comes first


## 5. Exports for the paper

*Write the queue (and any figures I want to reuse) to `work/outputs/` and `work/figures/` — the paper builds on these exact files.*

- `content_action_queue.csv` — the full ranked queue, with the baseline rule as comparison column. Stays out of git by design; the notebook regenerates it.
- `playbook_metrics.json` — the receipts: grouped metrics, memorization gap, queue composition, decay buckets, monitoring thresholds. Committed.
- Four figures in `work/figures/` — committed, reused in the paper.

In [7]:
export_cols = [
    'rank', 'content_id', 'client_id', 'content_type', 'action', 'reason_code',
    'model_score', 'baseline_score', 'impressions_90d', 'days_since_last_update',
    'avg_position', 'ctr', 'is_declining_label',
]
queue_path = OUTPUTS_DIR / 'content_action_queue.csv'
queue[export_cols].to_csv(queue_path, index=False)

playbook_metrics = {
    'notebook': 'work/notebooks/w07_action_playbook.ipynb',
    'split': {'group': 'client_id', 'test_size': 0.2, 'seed': 42},
    'metrics': metrics,
    'queue': {
        'rows': int(len(queue)),
        'actions': queue['action'].value_counts().to_dict(),
        'reason_codes': queue['reason_code'].value_counts().to_dict(),
    },
    'decay_buckets': decay.reset_index().assign(
        staleness_bucket=lambda d: d['staleness_bucket'].astype(str)
    ).to_dict(orient='records'),
    'guardrail_checks': guardrail_checks,
    'monitoring': monitoring,
}
metrics_path = OUTPUTS_DIR / 'playbook_metrics.json'
metrics_path.write_text(json.dumps(playbook_metrics, indent=2), encoding='utf-8')

# Figure 1: precision@k, grouped test
fig, ax = plt.subplots(figsize=(7, 4))
ks = ['p10', 'p20', 'p50']
x = np.arange(len(ks))
width = 0.25
ax.bar(x - width, [metrics['grouped'][f'baseline_{k}'] for k in ks], width, label='Baseline rule')
ax.bar(x, [metrics['grouped'][f'lr_{k}'] for k in ks], width, label='Logistic Regression')
ax.bar(x + width, [metrics['grouped'][f'rf_{k}'] for k in ks], width, label='Random Forest')
ax.set_xticks(x)
ax.set_xticklabels(['P@10', 'P@20', 'P@50'])
ax.set_ylabel('Precision')
ax.set_title('Precision at k — client-grouped test set')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_precision_at_k.png', dpi=150)
plt.close(fig)

# Figure 2: memorization gap
fig, ax = plt.subplots(figsize=(6, 4))
models = ['LR', 'RF']
random_aucs = [metrics['random']['lr_auc'], metrics['random']['rf_auc']]
grouped_aucs = [metrics['grouped']['lr_auc'], metrics['grouped']['rf_auc']]
x = np.arange(len(models))
ax.bar(x - 0.18, random_aucs, 0.36, label='Random split')
ax.bar(x + 0.18, grouped_aucs, 0.36, label='Grouped (honest)')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('AUC')
ax.set_ylim(0.4, 0.85)
ax.set_title('The memorization gap — random looks better than it is')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_memorization_gap.png', dpi=150)
plt.close(fig)

# Figure 3: decline by staleness, with n on each bar
fig, ax = plt.subplots(figsize=(7, 4))
buckets = decay.index.astype(str)
rates = decay['decline_rate'].values
counts = decay['pages'].values
bars = ax.bar(buckets, rates, color=['#4c78a8', '#f58518', '#bab0ac', '#bab0ac'])
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'n={n:,}', ha='center', fontsize=9)
ax.axhline(y.mean(), color='red', linestyle='--', linewidth=1, label=f'base rate {y.mean():.1%}')
ax.set_ylabel('Decline rate')
ax.set_title('Decay is not monotonic — and the old buckets are thin')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_decline_by_staleness.png', dpi=150)
plt.close(fig)

# Figure 4: queue composition
fig, ax = plt.subplots(figsize=(6, 4))
action_counts = queue['action'].value_counts()
ax.bar(action_counts.index, action_counts.values,
       color=[{'NO_ACTION': '#bab0ac', 'REVIEW': '#4c78a8',
               'REOPTIMIZE': '#f58518', 'REFRESH': '#e45756'}[a] for a in action_counts.index])
ax.set_ylabel('Pages')
ax.set_title('Queue composition by action (all 30k pages)')
for i, (action, count) in enumerate(action_counts.items()):
    ax.text(i, count + 200, f'{count:,}', ha='center', fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_queue_actions.png', dpi=150)
plt.close(fig)

print('Exports written:')
for path in [queue_path, metrics_path,
             FIGURES_DIR / 'fig_precision_at_k.png',
             FIGURES_DIR / 'fig_memorization_gap.png',
             FIGURES_DIR / 'fig_decline_by_staleness.png',
             FIGURES_DIR / 'fig_queue_actions.png']:
    print(f'  {path.relative_to(REPO_ROOT)}  ({path.stat().st_size:,} bytes)')

Exports written:
  work\outputs\content_action_queue.csv  (3,995,760 bytes)
  work\outputs\playbook_metrics.json  (2,390 bytes)
  work\figures\fig_precision_at_k.png  (32,066 bytes)
  work\figures\fig_memorization_gap.png  (29,635 bytes)
  work\figures\fig_decline_by_staleness.png  (34,648 bytes)
  work\figures\fig_queue_actions.png  (33,186 bytes)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.